In [49]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/bikrammaharjan/ai-usage-and-productivity/ai_adoption_productivity_2021_2026.csv
/kaggle/input/datasets/bikrammaharjan/ai-usage-and-productivity/user_level_ai_adoption.csv


In [50]:
!uv pip install pandas numpy scikit-learn xgboost joblib

Using Python 3.12.13 environment at: /usr
Checked 5 packages in 157ms


In [51]:
import os
import joblib
import glob
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [52]:
# =====================================================================
# 1. AUTOMATIC DATA LOADING FROM KAGGLE INPUT
# =====================================================================
# Automatically locate the CSV file inside Kaggle's /kaggle/input directory
csv_files = glob.glob('/kaggle/input/**/*.csv', recursive=True)

if not csv_files:
    raise FileNotFoundError("No CSV files found in /kaggle/input/. Please attach the dataset to your Kaggle Notebook.")

dataset_path = csv_files[0]
print(f"Loading dataset from: {dataset_path}")

df = pd.read_csv(dataset_path)
print(f"Dataset loaded successfully. Shape: {df.shape}")

Loading dataset from: /kaggle/input/datasets/bikrammaharjan/ai-usage-and-productivity/ai_adoption_productivity_2021_2026.csv
Dataset loaded successfully. Shape: (402, 6)


In [53]:
# Standardize column headers (lower case, strip whitespace)
df.columns = df.columns.str.strip().str.lower()

In [54]:
# =====================================================================
# 2. AUTO-DETECT TARGET & FEATURE COLUMNS
# =====================================================================
# Target keyword priority matching
potential_targets = [c for c in df.columns if any(kw in c for kw in ['productiv', 'gain', 'impact', 'change', 'score', 'rate'])]

if potential_targets:
    target_col = potential_targets[0]
else:
    # Fall back to the last numerical column as the target
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        target_col = num_cols[-1]
    else:
        raise ValueError("Could not find a valid numeric target column in your CSV.")

print(f"\n>>> Selected Target Column: '{target_col}'")


>>> Selected Target Column: 'productivity gain (%)'


In [55]:
# Clean dataset of missing targets
df = df.dropna(subset=[target_col])

In [56]:
# Auto-separate categorical and numerical features
feature_cols = [c for c in df.columns if c != target_col and 'id' not in c and 'key' not in c]

num_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
cat_features = df[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Detected Numerical Features ({len(num_features)}): {num_features}")
print(f"Detected Categorical Features ({len(cat_features)}): {cat_features}")

X = df[cat_features + num_features]
y = df[target_col]

Detected Numerical Features (2): ['global active users (millions)', 'average tokens/user/day']
Detected Categorical Features (3): ['yearmonth', 'industry', 'primary use case']


In [57]:
# =====================================================================
# 3. BUILD PREPROCESSING & MODEL PIPELINE
# =====================================================================
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1))
])

In [58]:
# =====================================================================
# 4. TRAIN AND EVALUATE
# =====================================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining Random Forest Regressor...")
model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)
print("\n--- Model Performance Metrics ---")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE:      {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE:     {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")


Training Random Forest Regressor...

--- Model Performance Metrics ---
R² Score: 0.9733
MAE:      1.9812
RMSE:     2.7191


In [59]:
# =====================================================================
# 5. EXPORT model.pkl BUNDLE
# =====================================================================
cat_encoder = model_pipeline.named_steps['preprocessor'].named_transformers_['cat']
encoded_cat_names = cat_encoder.get_feature_names_out(cat_features).tolist() if cat_features else []
all_feature_names = num_features + encoded_cat_names

importances = model_pipeline.named_steps['regressor'].feature_importances_
feature_importance_dict = dict(zip(all_feature_names, importances))

export_bundle = {
    'pipeline': model_pipeline,
    'target_column': target_col,
    'numerical_features': num_features,
    'categorical_features': cat_features,
    'feature_importances': feature_importance_dict,
    'categories': {
        col: list(df[col].dropna().unique()) for col in cat_features
    }
}

output_filename = '/kaggle/working/model.pkl'
joblib.dump(export_bundle, output_filename)

print(f"\nSuccessfully exported '{output_filename}'!")


Successfully exported '/kaggle/working/model.pkl'!
